# Phase 4 — Fine-tuning Stage 1 (spectral sharpening)

**Objectif** : fine-tuner le GNN/RCN/regression_head du 9-node depuis `epoch_best_stage1.pth`
en ajoutant `high_k_rapsd_loss` (λ=0.05, k_min=20) à la MSE existante.

**Protocole** :
1. Cell 5 — sonde baseline : `std(μ_HR)` et `A_dag` norm AVANT fine-tuning
2. Cell 6 — **1 epoch test** (`lambda_spectral_highk=0.05`)
3. Cell 7 — sonde post-epoch : `std(μ_HR)` ↑ ? `A_dag` stable ? → verdict GO/NO-GO
4. Cell 8 — si GO : lancer 4 epochs supplémentaires (total 5)

**Checkpoint source** : `oracle_9node/seed_42/epoch_best_stage1.pth`  
**Output** : `oracle_9node/seed_42/epoch_best_stage1_spectral.pth`

In [ ]:
# === Cell 1 : Bootstrap (Colab) ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab — Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')

In [ ]:
# === Cell 2 : Imports + constantes ===
import json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from omegaconf import OmegaConf
from torch.optim import Adam

# --- Chemins Drive ---
DRIVE_ROOT  = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N   = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_IN     = ORACLE_9N / 'epoch_best_stage1.pth'       # checkpoint source
CKPT_OUT    = ORACLE_9N / 'epoch_best_stage1_spectral.pth'  # sortie fine-tuning
PROBE_JSON  = ORACLE_9N / 'phase4_probe_metrics.json'

assert CKPT_IN.exists(), f'Checkpoint introuvable : {CKPT_IN}'
print(f'[Cell 2] Checkpoint source   : {CKPT_IN}')
print(f'[Cell 2] Checkpoint sortie   : {CKPT_OUT}')

# --- Hyperparams fine-tuning ---
FINETUNE_LR             = 1e-4
LAMBDA_SPECTRAL_HIGHK   = 0.05   # poids loss spectrale haute fréquence
K_HIGHK_MIN             = 20     # wavenumber minimal (~8-9 km sur grille 172×179)
N_EPOCHS_TEST           = 1      # Cell 6 : test 1 epoch
N_EPOCHS_FULL           = 4      # Cell 8 : 4 epochs supplémentaires si GO

# --- Seed ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'[Cell 2] λ_spectral={LAMBDA_SPECTRAL_HIGHK}  k_min={K_HIGHK_MIN}  LR={FINETUNE_LR}')

In [ ]:
# === Cell 3 : Config + pipeline 9-node + dataloaders ===
# Identique au Cell 2+3 de st_cdgm_path_c_option_c_9node.ipynb
import os
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True

GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)
CONFIG.training.batch_size = GPU_PROFILE['batch_size']
CONFIG.training.use_amp    = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

# Path C+ scalar overrides
ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# 9-node : inject humid metapaths
_OC9 = OmegaConf
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
_humid_mp = [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]
OmegaConf.set_struct(CONFIG, False)
for _m in _humid_mp:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(_OC9.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- K9 temporal split ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

# --- Data paths ---
_ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN            = int(CONFIG.data.seq_len)
BASELINE_STRATEGY  = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR    = int(CONFIG.data.baseline_factor)
NORMALIZE          = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY  = str(CONFIG.data.nan_fill_strategy)

_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

BATCH_SIZE   = int(CONFIG.training.batch_size)
NUM_WORKERS  = int(CONFIG.training.num_workers)
PIN_MEMORY   = bool(torch.cuda.is_available())

# --- Pipeline ---
pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print(f'[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

_loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kwargs['persistent_workers'] = True
    _loader_kwargs['prefetch_factor'] = 2

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kwargs)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kwargs)

# --- Builder (9-node) ---
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder ready  dyn={builder.dynamic_node_types}')

# --- iterate_batches + convert_sample_to_batch (9-node) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Cell 3] DEVICE={DEVICE}')

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, _VI[f'q_{lev}']]
        u = lr0[:, _VI[f'u_{lev}']]
        v = lr0[:, _VI[f'v_{lev}']]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt_nodes = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = lr0[:, _Q_IDX] if _Q_IDX else lr0
            elif nt == 'W500': dynamic_features[nt] = lr0[:, _W_IDX] if _W_IDX else lr0
            elif nt == 'IVT':  dynamic_features[nt] = _ivt_nodes
            else:              dynamic_features[nt] = lr0
    else:
        dynamic_features = {nt: lr0 for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {'lr': lr_tensor, 'residual': sample['residual'],
            'baseline': sample.get('baseline'), 'hetero': hetero,
            'time': sample.get('time')}

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

# Probe a sample to get channel dims
_probe = next(iter(train_dataset))
RCN_DRIVER_DIM = _probe['lr'].shape[1]
hr_channels    = _probe['residual'].shape[1]
print(f'[Cell 3] RCN_DRIVER_DIM={RCN_DRIVER_DIM}  hr_channels={hr_channels}')

In [ ]:
# === Cell 4 : Build stack 9-node + charger epoch_best_stage1.pth + geler A_dag ===
from omegaconf import OmegaConf as _OC
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    SpatialConditioningProjector, CausalConditioningProjector,
    HRTargetIdentifiabilityHead,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.models.regression_head import GraphToGridDecoder

def _build_encoder(builder, CONFIG, DEVICE):
    allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
    encoder_configs = []
    for _mp in CONFIG.encoder.metapaths:
        if _mp.src in allowed_nodes and _mp.target in allowed_nodes:
            encoder_configs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_mp.src, _mp.relation, _mp.target),
                pool=_mp.get('pool', 'mean'),
            ))
    if pipeline.get_static_dataset() is not None:
        encoder_configs.append(IntelligibleVariableConfig(
            name='static', meta_path=('SP_HR', 'causes', 'GP850'), pool='mean',
        ))
    encoder = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(DEVICE)
    return encoder, len(encoder_configs)

# Build fresh stack (random weights, then we overwrite with checkpoint)
torch.manual_seed(SEED); np.random.seed(SEED)
encoder, num_vars = _build_encoder(builder, CONFIG, DEVICE)

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=RCN_DRIVER_DIM,
    reconstruction_dim=RCN_DRIVER_DIM,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=int(CONFIG.diffusion.height),
    hr_w=int(CONFIG.diffusion.width),
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

print(f'[Cell 4] Stack instancié : {num_vars} vars, '
      f'encoder={sum(p.numel() for p in encoder.parameters()):,} params')

# --- Charger epoch_best_stage1.pth ---
def _load_sd(module, sd_key, ck):
    sd = ck.get(sd_key)
    if sd is None:
        print(f'  WARN {sd_key} absent du checkpoint')
        return
    if isinstance(sd, dict) and '_orig_mod' in next(iter(sd), ''):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    module.load_state_dict(sd, strict=True)

ck = torch.load(CKPT_IN, map_location=DEVICE, weights_only=False)
_load_sd(encoder,         'encoder_state_dict',         ck)
_load_sd(rcn_cell,        'rcn_cell_state_dict',         ck)
_load_sd(regression_head, 'regression_head_state_dict',  ck)
print(f'[Cell 4] Checkpoint chargé  (epoch={ck.get("epoch","?")})')

# --- Geler A_dag : le DAG est parfait (skeleton_F1=1.0), on ne le modifie pas ---
_rcn_core = rcn_cell
if hasattr(_rcn_core, '_orig_mod'):   # compiled
    _rcn_core = _rcn_core._orig_mod
if hasattr(_rcn_core, 'A_dag'):
    _rcn_core.A_dag.requires_grad_(False)
    _A_dag_ref = _rcn_core.A_dag.detach().clone()
    print(f'[Cell 4] A_dag gelé  shape={tuple(_rcn_core.A_dag.shape)}  '
          f'norm={_A_dag_ref.norm().item():.4f}')
else:
    _A_dag_ref = None
    print('[Cell 4] WARN : A_dag introuvable dans rcn_cell — vérifier architecture')

# Optimizer sur les paramètres NON gelés
_trainable = [p for p in list(encoder.parameters()) + list(rcn_cell.parameters())
              + list(regression_head.parameters()) if p.requires_grad]
optimizer_ft = Adam(_trainable, lr=FINETUNE_LR)
print(f'[Cell 4] Paramètres entraînables : {sum(p.numel() for p in _trainable):,}')

In [ ]:
# === Cell 5 : Sonde baseline — std(μ_HR) et A_dag norm AVANT fine-tuning ===

def _val_probe(encoder, rcn_runner, regression_head, val_dl, builder, device, label=''):
    """Forward pass sur val set, retourne std(mu_HR) + RMSE."""
    encoder.eval(); rcn_runner.cell.eval(); regression_head.eval()
    mu_list, sq_err_list, mask_list = [], [], []
    with torch.no_grad():
        for batch_list in iterate_batches(val_dl, builder, device):
            for batch in batch_list:
                lr   = batch['lr'].to(device)
                res  = batch['residual'].to(device) if batch['residual'] is not None else None
                het  = batch['hetero']
                node_emb = encoder(het)
                H_t      = rcn_runner(node_emb, lr)
                mu_HR    = regression_head(H_t, het)   # [B,1,H,W] log1p space
                mu_list.append(mu_HR.cpu())
                if res is not None:
                    sq_err_list.append((mu_HR - res.to(device)).pow(2).cpu())
    all_mu = torch.cat(mu_list, dim=0)
    std_mu = all_mu.std().item()
    rmse   = sq_err_list[0].mean().sqrt().item() if sq_err_list else float('nan')
    if sq_err_list:
        rmse = torch.cat(sq_err_list).mean().sqrt().item()
    print(f'[probe {label}]  std(μ_HR)={std_mu:.4f}  RMSE_s1={rmse:.5f}')
    return {'std_mu_HR': std_mu, 'rmse_s1': rmse}

print('=== Sonde AVANT fine-tuning ===')
metrics_before = _val_probe(encoder, rcn_runner, regression_head,
                             val_dataloader, builder, DEVICE, label='BEFORE')

if _A_dag_ref is not None:
    print(f'A_dag norm (référence) : {_A_dag_ref.norm().item():.4f}')

In [ ]:
# === Cell 6 : 1 epoch test fine-tuning ===
from st_cdgm.training.training_loop import train_epoch_stage1
from path_c_plus.scripts.option_c_helpers import schedule_lambdas, DEFAULT_HYPERPARAMS

# Paramètres Stage 1 (mêmes que le run original, sauf lambda_spectral_highk)
_ts = CONFIG.two_stage
_s1 = _ts.stage1

# Schedule lambdas à l'epoch 15 (fin du training original)
sched = schedule_lambdas(
    epoch=15, total_epochs=15,
    lambda_l1_start=PATHCPLUS_HYPERPARAM_OVERRIDES.get('lambda_l1_start', DEFAULT_HYPERPARAMS['lambda_l1_start']),
    lambda_l1_end=PATHCPLUS_HYPERPARAM_OVERRIDES.get('lambda_l1_end', DEFAULT_HYPERPARAMS['lambda_l1_end']),
    dag_gate_warmup_start_epoch=None,
    dag_gate_warmup_end_epoch=None,
)
print(f'[Cell 6] lambda_l1={sched["lambda_l1"]:.4f}  dag_grad_gate={sched["dag_grad_gate"]:.3f}')

encoder.train(); rcn_runner.cell.train(); regression_head.train()

for ft_epoch in range(1, N_EPOCHS_TEST + 1):
    print(f'\n--- Fine-tuning epoch {ft_epoch}/{N_EPOCHS_TEST} ---')
    metrics = train_epoch_stage1(
        encoder=encoder,
        rcn_runner=rcn_runner,
        regression_head=regression_head,
        optimizer=optimizer_ft,
        data_loader=train_dataloader,
        device=DEVICE,
        epoch_idx=16 + ft_epoch,    # continue from epoch 15
        lambda_reg=float(_s1.lambda_reg),
        beta_rec=float(_s1.beta_rec),
        gamma_dag_max=float(_s1.gamma_dag_max),
        gamma_dag_warmup_epochs=int(_s1.gamma_dag_warmup_epochs),
        lambda_l1=sched['lambda_l1'],
        lambda_dag_prior=float(_s1.lambda_dag_prior),
        dag_grad_gate_value=0.0,    # A_dag gelé → pas de gradient DAG
        abort_on_collapse=False,    # A_dag gelé, pas de risque de collapse
        dag_floor_projection=False, # idem
        gradient_clipping=float(_s1.gradient_clipping) if _s1.get('gradient_clipping') else None,
        dag_method=str(_s1.get('dag_method', 'dagma')),
        use_amp=bool(CONFIG.training.use_amp),
        # === NOUVEAU : perte spectrale ===
        lambda_spectral_highk=LAMBDA_SPECTRAL_HIGHK,
        k_highk_min=K_HIGHK_MIN,
    )
    print(f'  loss_total={metrics.get("loss_total",float("nan")):.5f}  '
          f'loss_reg={metrics.get("loss_reg",float("nan")):.5f}  '
          f'loss_spec={metrics.get("loss_spectral_highk",float("nan")):.5f}')

print('\n[Cell 6] Fine-tuning test epoch terminé.')

In [ ]:
# === Cell 7 : Sonde post-epoch + vérification A_dag + verdict GO / NO-GO ===
print('=== Sonde APRÈS fine-tuning (1 epoch) ===')
metrics_after = _val_probe(encoder, rcn_runner, regression_head,
                            val_dataloader, builder, DEVICE, label='AFTER')

# Vérification A_dag : doit rester identique (gelé)
if _A_dag_ref is not None:
    A_now = _rcn_core.A_dag.detach()
    dag_drift = (A_now - _A_dag_ref).abs().max().item()
    print(f'A_dag max drift (doit être 0) : {dag_drift:.2e}')
    assert dag_drift < 1e-6, 'A_dag a bougé ! Le freeze ne fonctionne pas.'

# Verdict
std_before = metrics_before['std_mu_HR']
std_after  = metrics_after['std_mu_HR']
gain_pct   = 100 * (std_after - std_before) / (std_before + 1e-9)

print(f'\n{"="*60}')
print(f'std(μ_HR) BEFORE : {std_before:.4f}')
print(f'std(μ_HR) AFTER  : {std_after:.4f}   (gain={gain_pct:+.1f}%)')
print(f'RMSE_s1  BEFORE  : {metrics_before["rmse_s1"]:.5f}')
print(f'RMSE_s1  AFTER   : {metrics_after["rmse_s1"]:.5f}')

GO = (std_after > std_before * 1.05) and (metrics_after['rmse_s1'] <= metrics_before['rmse_s1'] * 1.10)

if GO:
    print('\n✓ VERDICT GO — std(μ_HR) augmente, RMSE_s1 reste contrôlé.')
    print('  Lancer Cell 8 pour 4 epochs supplémentaires.')
    # Sauvegarder le checkpoint de test
    _ck_test = {
        'schema_version': 1,
        'ft_epoch': N_EPOCHS_TEST,
        'encoder_state_dict': encoder.state_dict(),
        'rcn_cell_state_dict': rcn_cell.state_dict(),
        'regression_head_state_dict': regression_head.state_dict(),
        'optimizer_ft_state_dict': optimizer_ft.state_dict(),
        'metrics_before': metrics_before,
        'metrics_after_ep1': metrics_after,
        'lambda_spectral_highk': LAMBDA_SPECTRAL_HIGHK,
        'k_highk_min': K_HIGHK_MIN,
    }
    CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(_ck_test, CKPT_OUT)
    print(f'  Checkpoint sauvegardé : {CKPT_OUT}')
else:
    print(f'\n✗ VERDICT NO-GO.')
    if std_after <= std_before * 1.05:
        print(f'  std(μ_HR) n\'augmente pas assez (+{gain_pct:.1f}% < +5%).')
        print('  Essayer lambda_spectral_highk=0.2 ou k_highk_min=10.')
    if metrics_after['rmse_s1'] > metrics_before['rmse_s1'] * 1.10:
        print('  RMSE_s1 a augmenté > +10% : la loss spectrale est trop forte.')
        print('  Essayer lambda_spectral_highk=0.01.')

# Sauvegarder JSON pour traçabilité
probe_out = {
    'lambda_spectral_highk': LAMBDA_SPECTRAL_HIGHK,
    'k_highk_min': K_HIGHK_MIN,
    'finetune_lr': FINETUNE_LR,
    'n_epochs_test': N_EPOCHS_TEST,
    'before': metrics_before,
    'after_ep1': metrics_after,
    'std_gain_pct': gain_pct,
    'verdict': 'GO' if GO else 'NO-GO',
}
PROBE_JSON.write_text(json.dumps(probe_out, indent=2))
print(f'\nSauvegardé : {PROBE_JSON}')

In [ ]:
# === Cell 8 : FULL fine-tuning — 4 epochs supplémentaires (si VERDICT GO) ===
# À lancer UNIQUEMENT si Cell 7 a affiché VERDICT GO.
#
# Recharge le checkpoint de test (Cell 7) et continue 4 epochs.
# Output final : epoch_best_stage1_spectral.pth (mis à jour à chaque epoch).

assert CKPT_OUT.exists(), 'CKPT_OUT absent — Cell 7 doit afficher GO avant de lancer Cell 8.'

ck_resume = torch.load(CKPT_OUT, map_location=DEVICE, weights_only=False)
_load_sd(encoder,         'encoder_state_dict',         ck_resume)
_load_sd(rcn_cell,        'rcn_cell_state_dict',         ck_resume)
_load_sd(regression_head, 'regression_head_state_dict',  ck_resume)
optimizer_ft.load_state_dict(ck_resume['optimizer_ft_state_dict'])
ft_epoch_start = ck_resume.get('ft_epoch', N_EPOCHS_TEST)
print(f'[Cell 8] Reprise depuis ft_epoch={ft_epoch_start}')

best_rmse = metrics_before['rmse_s1']   # objectif : battre RMSE baseline
for ft_ep in range(ft_epoch_start + 1, ft_epoch_start + N_EPOCHS_FULL + 1):
    encoder.train(); rcn_runner.cell.train(); regression_head.train()
    print(f'\n--- Fine-tuning epoch {ft_ep}/{ft_epoch_start + N_EPOCHS_FULL} ---')
    m = train_epoch_stage1(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        optimizer=optimizer_ft, data_loader=train_dataloader,
        device=DEVICE, epoch_idx=16 + ft_ep,
        lambda_reg=float(_s1.lambda_reg), beta_rec=float(_s1.beta_rec),
        gamma_dag_max=float(_s1.gamma_dag_max),
        gamma_dag_warmup_epochs=int(_s1.gamma_dag_warmup_epochs),
        lambda_l1=sched['lambda_l1'],
        lambda_dag_prior=float(_s1.lambda_dag_prior),
        dag_grad_gate_value=0.0,
        abort_on_collapse=False, dag_floor_projection=False,
        gradient_clipping=float(_s1.gradient_clipping) if _s1.get('gradient_clipping') else None,
        dag_method=str(_s1.get('dag_method', 'dagma')),
        use_amp=bool(CONFIG.training.use_amp),
        lambda_spectral_highk=LAMBDA_SPECTRAL_HIGHK,
        k_highk_min=K_HIGHK_MIN,
    )
    print(f'  loss={m.get("loss_total",float("nan")):.5f}  '
          f'spec={m.get("loss_spectral_highk",float("nan")):.5f}')

    probe = _val_probe(encoder, rcn_runner, regression_head,
                       val_dataloader, builder, DEVICE, label=f'ep{ft_ep}')

    _ck_save = {
        'schema_version': 1, 'ft_epoch': ft_ep,
        'encoder_state_dict': encoder.state_dict(),
        'rcn_cell_state_dict': rcn_cell.state_dict(),
        'regression_head_state_dict': regression_head.state_dict(),
        'optimizer_ft_state_dict': optimizer_ft.state_dict(),
        'metrics_before': metrics_before, f'metrics_ep{ft_ep}': probe,
        'lambda_spectral_highk': LAMBDA_SPECTRAL_HIGHK, 'k_highk_min': K_HIGHK_MIN,
    }
    torch.save(_ck_save, CKPT_OUT)
    print(f'  Sauvegardé -> {CKPT_OUT}')

    if probe['rmse_s1'] < best_rmse:
        best_rmse = probe['rmse_s1']
        _best_path = ORACLE_9N / 'epoch_best_stage1_spectral_best.pth'
        torch.save(_ck_save, _best_path)
        print(f'  Nouveau best RMSE_s1={best_rmse:.5f} -> {_best_path}')

print('\n[Cell 8] Fine-tuning complet.')
print('Prochaine étape : relancer phase3_mu_HR_probe.ipynb avec CKPT_9N pointant')
print(f'vers epoch_best_stage1_spectral_best.pth pour vérifier si F1@p99 dépasse 0.46.')